In [4]:
from pathlib import Path
import xarray as xr
import sys

raiz = Path.cwd()
if not (raiz / "scripts/processing/spatial.py").is_file():
    raiz = raiz.parent  # Si Jupyter se inició dentro de notebooks/

sys.path.insert(0, str(raiz / "scripts/processing"))
from spatial import inspect_geometries, read_codes

_, (oeste, sur, este, norte) = inspect_geometries(set(read_codes()))

archivo = (
        raiz / "data/raw/precipitation/chirps/2025/"
               "chirps-v3.0.rnl.2025.days_p05.nc"
)

ds = xr.open_dataset(archivo, engine="netcdf4")
display(ds)


<xarray.Dataset> Size: 25GB
Dimensions:    (time: 365, latitude: 2400, longitude: 7200)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2025-01-01 2025-01-02 ... 2025-12-31
  * latitude   (latitude) float32 10kB -59.97 -59.92 -59.88 ... 59.92 59.97
  * longitude  (longitude) float32 29kB -180.0 -179.9 -179.9 ... 179.9 180.0
Data variables:
    precip     (time, latitude, longitude) float32 25GB ...
Attributes: (12/14)
    Conventions:       CF-1.6
    title:             CHIRPS Version 3.0
    history:           created by Climate Hazards Center
    version:           Version 3.0
    data_created:      2026-08-17
    creator_name:      Pete Peterson
    ...                ...
    documentation:     http://pubs.usgs.gov/ds/832/
    reference:         Funk, C.C., Peterson, P.J., Landsfeld, M.F., Pedreros,...
    acknowledgements:  The Climate Hazards Center InfraRed Precipitation with...
    ftp_url:           ftp://chg-ftpout.geog.ucsb.edu/pub/chg/products/CHIRPS...
    website:           http://chg.geog.ucsb.edu/data/chirps/index.html
    faq:               http://chg-wiki.geog.ucsb.edu/wiki/CHIRPS_FAQ

In [3]:
lluvia = (
    ds["precip"]
    .sel(time="2025-01-01")
    .sel(latitude=4.6, longitude=-74.1, method="nearest")
)

print(f"{lluvia.item():.2f} mm")

2.78 mm


In [5]:
# Un margen de 0,05° incluye las celdas que tocan el borde.
precip_colombia = ds["precip"].sel(
    latitude=slice(sur - 0.05, norte + 0.05),
    longitude=slice(oeste - 0.05, este + 0.05),
)

print(precip_colombia.sizes)

# Cargar únicamente los primeros siete días del área recortada.
bloque = precip_colombia.isel(time=slice(0, 7)).to_numpy()
print(bloque.shape, f"{bloque.nbytes / 2**20:.2f} MiB")

Frozen({'time': 365, 'latitude': 355, 'longitude': 282})
(7, 355, 282) 2.67 MiB


In [6]:
from epidemiological_weeks import (
    calendar_years_needed,
    day_to_week,
    weeks_in_range,
)
from spatial import build_weights, inspect_geometries, read_codes

# Calendario compartido con el procesamiento de ERA5.
semanas = weeks_in_range(2007, 2025)
indice_semana_por_dia = day_to_week(semanas)
anios_calendario = calendar_years_needed(2007, 2025)

# Municipios MGN 2025 comprobados contra DIVIPOLA.
nombres_municipios = read_codes()
codigos, limites = inspect_geometries(set(nombres_municipios))

# Guardar los ejes del recorte: los índices de los pesos se referirán
# a este arreglo recortado, no a la grilla global.
latitudes = precip_colombia.latitude.to_numpy().copy()
longitudes = precip_colombia.longitude.to_numpy().copy()

(
    indice_lat,
    indice_lon,
    indice_municipio,
    area_interseccion_m2,
    area_total_m2,
) = build_weights(latitudes, longitudes, codigos, limites)

print(f"{len(semanas)} semanas; {len(codigos)} municipios")
print(f"{len(area_interseccion_m2)} intersecciones municipio–celda")
print(f"Años calendario necesarios: {anios_calendario.start}–{anios_calendario.stop - 1}")

992 semanas; 1103 municipios
48222 intersecciones municipio–celda
Años calendario necesarios: 2006–2026


In [7]:
indice_lat

array([209, 209, 209, ..., 193, 193, 194], shape=(48222,), dtype=int32)

In [8]:
import numpy as np

def agregar_dia_municipios(campo):
    # Un valor por cada intersección municipio–celda.
    lluvia = campo[indice_lat, indice_lon]
    valida = np.isfinite(lluvia) & (lluvia >= 0)

    area_valida = np.bincount(
        indice_municipio,
        weights=np.where(valida, area_interseccion_m2, 0.0),
        minlength=len(codigos),
    )
    lluvia_por_area = np.bincount(
        indice_municipio,
        weights=np.where(valida, lluvia, 0.0) * area_interseccion_m2,
        minlength=len(codigos),
    )

    media_mm = np.divide(
        lluvia_por_area,
        area_valida,
        out=np.full(len(codigos), np.nan),
        where=area_valida > 0,
    )
    fraccion_area = np.divide(
        area_valida,
        area_total_m2,
        out=np.zeros(len(codigos)),
        where=area_total_m2 > 0,
    )

    return media_mm, fraccion_area

In [10]:
import pandas as pd
resultados = []
oeste, sur, este, norte = limites

recorte_anio = precip_colombia

assert np.array_equal(recorte_anio.latitude.values, latitudes)
assert np.array_equal(recorte_anio.longitude.values, longitudes)

for inicio in range(0, recorte_anio.sizes["time"], 7):
    bloque = recorte_anio.isel(time=slice(inicio, inicio + 7))
    fechas = pd.to_datetime(bloque.time.values).date

    for fecha, campo in zip(fechas, bloque.to_numpy()):
        media_mm, fraccion_area = agregar_dia_municipios(campo)

        resultados.append(pd.DataFrame({
            "codigo_municipio": codigos,
            "fecha": fecha,
            "precipitacion_diaria_mm": media_mm,
            "fraccion_area_con_datos": fraccion_area,
        }))

precipitacion_diaria = pd.concat(resultados, ignore_index=True)
display(precipitacion_diaria.head())

,codigo_municipio,fecha,precipitacion_diaria_mm,fraccion_area_con_datos
0,05001,2025-01-01,1.045557,1.0
1,05002,2025-01-01,2.533289,1.0
2,05004,2025-01-01,6.556684,1.0
3,05021,2025-01-01,3.408358,1.0
4,05030,2025-01-01,3.502667,1.0


In [11]:
fechas = pd.to_datetime(precipitacion_diaria["fecha"])

for etiqueta, fecha in (
        ("Más antigua", fechas.min()),
        ("Más reciente", fechas.max()),
):
    filas = precipitacion_diaria.loc[fechas.eq(fecha)]
    print(f"{etiqueta}: {fecha.date()} — {len(filas)} filas")
    display(filas.head())

Más antigua: 2025-01-01 — 1103 filas


,codigo_municipio,fecha,precipitacion_diaria_mm,fraccion_area_con_datos
0,05001,2025-01-01,1.045557,1.0
1,05002,2025-01-01,2.533289,1.0
2,05004,2025-01-01,6.556684,1.0
3,05021,2025-01-01,3.408358,1.0
4,05030,2025-01-01,3.502667,1.0


Más reciente: 2025-12-31 — 1103 filas


,codigo_municipio,fecha,precipitacion_diaria_mm,fraccion_area_con_datos
401492,05001,2025-12-31,0.810630,1.0
401493,05002,2025-12-31,0.173049,1.0
401494,05004,2025-12-31,0.506457,1.0
401495,05021,2025-12-31,1.459676,1.0
401496,05030,2025-12-31,1.167997,1.0


In [12]:

from datetime import timedelta
import pandas as pd

datos = precipitacion_diaria.copy()
datos["fecha"] = pd.to_datetime(datos["fecha"]).dt.date

if datos.duplicated(["codigo_municipio", "fecha"]).any():
    raise ValueError("Hay más de una fila por municipio y día")

datos["indice_semana"] = datos["fecha"].map(indice_semana_por_dia)
datos = datos.dropna(subset=["indice_semana"]).copy()
datos["indice_semana"] = datos["indice_semana"].astype(int)

semanal = (
    datos.groupby(["codigo_municipio", "indice_semana"], as_index=False)
    .agg(
        precipitacion_total_mm=(
            "precipitacion_diaria_mm",
            lambda valores: valores.sum(min_count=7),
        ),
        fraccion_area_con_datos=(
            "fraccion_area_con_datos",
            lambda valores: valores.fillna(0).sum() / 7,
        ),
        dias_con_datos=("precipitacion_diaria_mm", "count"),
    )
)

calendario = pd.DataFrame([
    {
        "indice_semana": i,
        "ano_epidemiologico": anio,
        "semana_epidemiologica": numero,
        "inicio_semana": inicio,
        "fin_semana": inicio + timedelta(days=6),
    }
    for i, (anio, numero, inicio) in enumerate(semanas)
])

precipitacion_semanal = (
    semanal.merge(calendario, on="indice_semana", validate="many_to_one")
    .drop(columns="indice_semana")
)

display(precipitacion_semanal.head())


,codigo_municipio,precipitacion_total_mm,fraccion_area_con_datos,dias_con_datos,ano_epidemiologico,semana_epidemiologica,inicio_semana,fin_semana
0,05001,NaN,0.571429,4,2025,1,2024-12-29,2025-01-04
1,05001,23.556770,1.000000,7,2025,2,2025-01-05,2025-01-11
2,05001,0.000000,1.000000,7,2025,3,2025-01-12,2025-01-18
3,05001,0.036198,1.000000,7,2025,4,2025-01-19,2025-01-25
4,05001,23.279111,1.000000,7,2025,5,2025-01-26,2025-02-01
